# 03 — State Legislative Activity and Policy Orientation


    **Research objective.** Describe the volume, timing, geographic distribution, and policy orientation of scored state data center bills since 2023.

    **Inputs.** `../data/processed/bills_scored.csv`.

    **Methods.** Use pandas aggregation and a bill-level event map. Color represents the 1–10 policy-orientation score with 5 as the neutral midpoint; black outlines identify observed advancement. State and quarterly summaries replace unstable state-specific regressions.

    **Outputs.** `bill_event_map.png`, `bill_state_summary_heatmap.png`, and state/quarter tables in `../output/`.

In [1]:
from __future__ import annotations

import json

import math

import re

from pathlib import Path

import matplotlib

import matplotlib.dates as mdates

import matplotlib.pyplot as plt

import numpy as np

import pandas as pd

from scipy.stats import spearmanr

from sklearn.compose import ColumnTransformer

from sklearn.decomposition import LatentDirichletAllocation

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from sklearn.model_selection import (
    GroupShuffleSplit,
    StratifiedKFold,
    cross_val_predict,
    train_test_split,
)

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder, StandardScaler

matplotlib.use("Agg")

SEED = 149

REGIONS = {
    "California": "West", "Oregon": "West",
    "Illinois": "Midwest", "Indiana": "Midwest", "Minnesota": "Midwest",
    "North Dakota": "Midwest", "Ohio": "Midwest",
    "Georgia": "South", "Kentucky": "South", "Maryland": "South",
    "Oklahoma": "South", "Texas": "South", "Virginia": "South",
    "Maine": "Northeast", "New York": "Northeast",
}

def save_table(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)
    print(f"Saved {len(frame):,} rows -> {path}")

def save_figure(fig: plt.Figure, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"Saved figure -> {path}")

def load_scored_bills(path: Path) -> pd.DataFrame:
    bills = pd.read_csv(path, low_memory=False)
    for column in ["first_action_date", "latest_action_date", "latest_passage_date"]:
        bills[column] = pd.to_datetime(bills[column], errors="coerce", utc=True)
    numeric = [
        "supportiveness_score", "supportiveness_confidence", "sponsor_count",
        "primary_sponsor_count", "passage_count", "passed_vote_event_count",
    ]
    for column in numeric:
        bills[column] = pd.to_numeric(bills[column], errors="coerce")
    bills["advanced"] = (
        bills["derived_status"].isin(["passed_chamber_or_legislature", "enacted", "vetoed"])
        | bills["passage_count"].fillna(0).gt(0)
        | bills["passed_vote_event_count"].fillna(0).gt(0)
    ).astype(int)
    bills["score_numeric"] = bills["supportiveness_score"]
    bills["introduction_year"] = bills["first_action_date"].dt.year.astype("Int64")
    bills["region"] = bills["state"].map(REGIONS).fillna("Other")
    return bills

def scored_analysis_sample(bills: pd.DataFrame) -> pd.DataFrame:
    mask = (
        bills["dc_relevant"].fillna("").str.lower().eq("yes")
        & bills["supportiveness_status"].fillna("").str.lower().eq("scored")
        & bills["score_numeric"].between(1, 10)
    )
    out = bills.loc[mask].copy()
    out["supportiveness_group"] = pd.cut(
        out["score_numeric"], bins=[0, 3, 6, 10],
        labels=["Restrictive (1-3)", "Neutral or mixed (4-6)", "Supportive (7-10)"],
    )
    return out

def run_bill_landscape(root: Path) -> dict:
    bills = scored_analysis_sample(load_scored_bills(root / "data/processed/bills_scored.csv"))
    print(f"Input relevant/scored rows: {len(bills):,}")
    print(f"Missing first action dates: {bills.first_action_date.isna().sum():,}")
    print(f"Duplicate bill IDs: {bills.bill_id.duplicated().sum():,}")

    dated = bills.dropna(subset=["first_action_date"]).copy()
    states = bills.groupby("state").size().sort_values().index.tolist()
    y_map = {state: i for i, state in enumerate(states)}
    rng = np.random.default_rng(SEED)
    y = dated.state.map(y_map).to_numpy() + rng.uniform(-0.18, 0.18, len(dated))
    fig, ax = plt.subplots(figsize=(13, 8))
    scatter = ax.scatter(
        dated.first_action_date, y, c=dated.score_numeric, cmap="RdYlGn", vmin=1, vmax=10,
        s=np.where(dated.advanced.eq(1), 58, 34), alpha=.72,
        edgecolors=np.where(dated.advanced.eq(1), "black", "none"), linewidths=.8,
    )
    ax.set_yticks(range(len(states)), states)
    ax.axvline(pd.Timestamp("2024-01-01", tz="UTC"), color="0.5", ls="--", lw=1)
    ax.set_xlabel("First recorded action date")
    ax.set_ylabel("State")
    ax.set_title(f"State Data Center Legislation by First Recorded Action Date, 2023–2026 (n={len(dated):,})\nBlack outlines indicate observed advancement; score 5 is the neutral or mixed midpoint")
    ax.grid(axis="x", alpha=.2)
    cbar = fig.colorbar(scatter, ax=ax, pad=.01)
    cbar.set_label("AI-assisted policy-orientation score (1 = restrictive; 10 = supportive)")
    fig.tight_layout()
    save_figure(fig, root / "output/figures/bill_event_map.png")

    bills["observed_bill_date"] = bills["first_action_date"].fillna(bills["latest_action_date"])
    bills["passed_at_least_one_chamber_flag"] = (
        bills["derived_status"].isin(["passed_chamber_or_legislature", "enacted", "vetoed"])
        | bills["passage_count"].fillna(0).gt(0)
    ).astype(int)
    summary = bills.groupby("state").agg(
        bill_count=("bill_id", "nunique"), median_supportiveness=("score_numeric", "median"),
        mean_supportiveness=("score_numeric", "mean"), advancement_rate=("advanced", "mean"),
        enacted=("derived_status", lambda x: int((x == "enacted").sum())),
        passed_at_least_one_chamber=("passed_at_least_one_chamber_flag", "sum"),
        first_observed_bill_date=("observed_bill_date", "min"), last_observed_bill_date=("observed_bill_date", "max"),
    ).reset_index()
    save_table(summary, root / "output/tables/bill_state_summary.csv")

    heat = summary.set_index("state")[["bill_count", "median_supportiveness", "advancement_rate"]].sort_values("bill_count")
    scaled = (heat - heat.min()) / (heat.max() - heat.min()).replace(0, 1)
    fig, ax = plt.subplots(figsize=(8.5, 8))
    im = ax.imshow(scaled, cmap="YlGnBu", aspect="auto", vmin=0, vmax=1)
    labels = [[f"{r.bill_count:.0f}", f"{r.median_supportiveness:.1f}", f"{100*r.advancement_rate:.1f}%"] for r in heat.itertuples()]
    for i in range(len(heat)):
        for j in range(3):
            ax.text(j, i, labels[i][j], ha="center", va="center", color="black", fontsize=9)
    ax.set_xticks(range(3), ["Bill count", "Median policy orientation", "Advancement rate"])
    ax.set_yticks(range(len(heat)), heat.index)
    ax.set_title(f"State-Level Legislative Activity and Policy Orientation (n={len(bills):,} scored bills)\nColor intensity is normalized within each column; annotations report original values")
    fig.colorbar(im, ax=ax, label="Within-column relative intensity", shrink=.8)
    fig.tight_layout()
    save_figure(fig, root / "output/figures/bill_state_summary_heatmap.png")

    quarterly = dated.assign(quarter=dated.first_action_date.dt.tz_localize(None).dt.to_period("Q").astype(str)).groupby("quarter").agg(
        new_bill_count=("bill_id", "nunique"), median_supportiveness=("score_numeric", "median"), advancement_rate=("advanced", "mean")
    ).reset_index()
    save_table(quarterly, root / "output/tables/bill_quarter_summary.csv")
    return {"scored_bills": len(bills), "dated_bills": len(dated), "states": len(summary)}

ROOT = Path("..")
print("Random seed:", SEED)

Random seed: 149


In [2]:
results = run_bill_landscape(ROOT)
results

Input relevant/scored rows: 395
Missing first action dates: 32
Duplicate bill IDs: 0


Saved figure -> ../output/figures/bill_event_map.png
Saved 15 rows -> ../output/tables/bill_state_summary.csv


Saved figure -> ../output/figures/bill_state_summary_heatmap.png
Saved 12 rows -> ../output/tables/bill_quarter_summary.csv


{'scored_bills': 395, 'dated_bills': 363, 'states': 15}

## Interpretation, inferential scope, and limitations

Counts describe the fifteen-state collected sample rather than all state legislation. Policy orientation is an AI-assisted ordinal measure, and event-map jitter serves only to display overlapping observations. Advancement is measured as of the collection date; bills introduced later have less time to advance. The figures are descriptive and do not estimate causal effects.